# Notebook 10 — Phase 2: Automated Testing with pytest




In [1]:
# Cell 1 — Find the project root and confirm model artifacts exist
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "models").exists() and (PROJECT_ROOT.parent / "models").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_PATH = PROJECT_ROOT / "models" / "fraud_detection_pipeline.pkl"
METADATA_PATH = PROJECT_ROOT / "models" / "fraud_detection_metadata.json"

print("Project root:", PROJECT_ROOT)
print("Model artifact exists:", MODEL_PATH.exists())
print("Metadata exists:", METADATA_PATH.exists())

if not MODEL_PATH.exists() or not METADATA_PATH.exists():
    raise FileNotFoundError("Open this notebook from your existing fraud-detection-system project.")


Project root: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system
Model artifact exists: True
Metadata exists: True


In [2]:
# Cell 2 — Install pytest and API dependencies
%pip install fastapi uvicorn httpx pytest joblib scikit-learn pandas numpy


Note: you may need to restart the kernel to use updated packages.


In [3]:
# Cell 3 — Create an importable FastAPI module (app.py)
# This keeps API code in one reusable place for local server runs and pytest.
app_code = r'''
from pathlib import Path
import json
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, ConfigDict
from typing import Literal

PROJECT_ROOT = Path(__file__).resolve().parent
MODEL_PATH = PROJECT_ROOT / "models" / "fraud_detection_pipeline.pkl"
METADATA_PATH = PROJECT_ROOT / "models" / "fraud_detection_metadata.json"

if not MODEL_PATH.exists() or not METADATA_PATH.exists():
    raise FileNotFoundError("Model artifacts not found under project_root/models.")

loaded_artifact = joblib.load(MODEL_PATH)
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))

model = loaded_artifact
if isinstance(loaded_artifact, dict):
    candidate_keys = ["model", "pipeline", "best_model", "best_estimator",
                      "estimator", "classifier", "fraud_detection_pipeline"]
    model = next(
        (loaded_artifact[key] for key in candidate_keys
         if key in loaded_artifact and hasattr(loaded_artifact[key], "predict_proba")),
        None
    )
    if model is None:
        candidates = [value for value in loaded_artifact.values()
                      if hasattr(value, "predict_proba")]
        if len(candidates) == 1:
            model = candidates[0]
        else:
            raise TypeError("Could not identify a unique predict_proba estimator in saved artifact.")

if not hasattr(model, "predict_proba"):
    raise TypeError("Loaded model does not provide predict_proba().")

feature_names = metadata.get("features") or metadata.get("feature_names")
if not feature_names and isinstance(loaded_artifact, dict):
    feature_names = loaded_artifact.get("features") or loaded_artifact.get("feature_names")
if not feature_names:
    feature_names = [
        "type", "amount", "oldbalanceOrg", "oldbalanceDest", "hour_of_day",
        "night_flag", "amount_to_origin_balance_ratio", "origin_balance_was_zero",
        "origin_txn_count_prior_steps"
    ]

threshold = float(metadata.get("threshold", 0.5))

app = FastAPI(title="Fraud Detection API", version="1.0.0")

class TransactionFeatures(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: Literal["PAYMENT", "TRANSFER", "CASH_OUT", "DEBIT", "CASH_IN"]
    amount: float = Field(ge=0)
    oldbalanceOrg: float = Field(ge=0)
    oldbalanceDest: float = Field(ge=0)
    hour_of_day: int = Field(ge=0, le=23)
    night_flag: int = Field(ge=0, le=1)
    amount_to_origin_balance_ratio: float = Field(ge=0)
    origin_balance_was_zero: int = Field(ge=0, le=1)
    origin_txn_count_prior_steps: int = Field(ge=0)

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": model is not None}

@app.get("/model-info")
def model_info():
    return {
        "model_type": type(model).__name__,
        "feature_count": len(feature_names),
        "features": feature_names,
        "threshold": threshold,
    }

@app.post("/predict")
def predict(transaction: TransactionFeatures):
    row = transaction.model_dump()
    missing = [name for name in feature_names if name not in row]
    if missing:
        raise HTTPException(status_code=422, detail=f"Missing required model features: {missing}")
    X = pd.DataFrame([{name: row[name] for name in feature_names}], columns=feature_names)
    try:
        probabilities = model.predict_proba(X)
        fraud_index = list(model.classes_).index(1)
        fraud_probability = float(probabilities[0][fraud_index])
        prediction = int(fraud_probability >= threshold)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Prediction failed: {type(exc).__name__}: {exc}")
    return {
        "is_fraud": prediction,
        "fraud_probability": fraud_probability,
        "threshold": threshold,
        "decision": "review_as_potential_fraud" if prediction else "not_flagged_by_model",
        "disclaimer": "Model output is a risk signal, not proof of fraud.",
    }
'''
app_path = PROJECT_ROOT / "app.py"
app_path.write_text(app_code.strip() + "\n", encoding="utf-8")
print("Created:", app_path)


Created: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\app.py


In [4]:
# Cell 4 — Create pytest tests and pytest configuration
tests_dir = PROJECT_ROOT / "tests"
tests_dir.mkdir(parents=True, exist_ok=True)

test_code = r'''
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

VALID_TRANSACTION = {
    "type": "TRANSFER",
    "amount": 1500.0,
    "oldbalanceOrg": 2000.0,
    "oldbalanceDest": 500.0,
    "hour_of_day": 14,
    "night_flag": 0,
    "amount_to_origin_balance_ratio": 0.75,
    "origin_balance_was_zero": 0,
    "origin_txn_count_prior_steps": 3,
}

def test_health_endpoint():
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json()["status"] == "ok"
    assert response.json()["model_loaded"] is True

def test_model_info_endpoint():
    response = client.get("/model-info")
    assert response.status_code == 200
    assert response.json()["feature_count"] == 9
    assert response.json()["threshold"] is not None

def test_valid_prediction_request():
    response = client.post("/predict", json=VALID_TRANSACTION)
    assert response.status_code == 200
    body = response.json()
    assert body["is_fraud"] in (0, 1)
    assert 0 <= body["fraud_probability"] <= 1
    assert 0 <= body["threshold"] <= 1

def test_negative_amount_is_rejected():
    bad_transaction = {**VALID_TRANSACTION, "amount": -1}
    response = client.post("/predict", json=bad_transaction)
    assert response.status_code == 422

def test_unexpected_field_is_rejected():
    bad_transaction = {**VALID_TRANSACTION, "unexpected_field": "not allowed"}
    response = client.post("/predict", json=bad_transaction)
    assert response.status_code == 422

def test_invalid_hour_is_rejected():
    bad_transaction = {**VALID_TRANSACTION, "hour_of_day": 24}
    response = client.post("/predict", json=bad_transaction)
    assert response.status_code == 422
'''
test_path = tests_dir / "test_api.py"
test_path.write_text(test_code.strip() + "\n", encoding="utf-8")

pytest_ini = PROJECT_ROOT / "pytest.ini"
pytest_ini.write_text(
    "[pytest]\n"
    "testpaths = tests\n"
    "python_files = test_*.py\n"
    "addopts = -ra\n",
    encoding="utf-8"
)

print("Created:", test_path)
print("Created:", pytest_ini)


Created: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\tests\test_api.py
Created: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\pytest.ini


In [5]:
# Cell 5 — Run pytest from the project root
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)
print("pytest exit code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError("pytest did not pass. Review the error output above before continuing.")


============================= test session starts =============================
platform win32 -- Python 3.14.8, pytest-9.1.1, pluggy-1.6.0 -- c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system
configfile: pytest.ini
testpaths: tests
plugins: anyio-4.15.1, platformdirs-4.13.0
collecting ... collected 6 items

tests/test_api.py::test_health_endpoint PASSED                           [ 16%]
tests/test_api.py::test_model_info_endpoint PASSED                       [ 33%]
tests/test_api.py::test_valid_prediction_request PASSED                  [ 50%]
tests/test_api.py::test_negative_amount_is_rejected PASSED               [ 66%]
tests/test_api.py::test_unexpected_field_is_rejected PASSED              [ 83%]
tests/test_api.py::test_invalid_hour_is_rejected PASSED                  [100%]

============================== warnings summary =========

In [6]:
# Cell 6 — Save a short test execution record
output_dir = PROJECT_ROOT / "notebooks" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
summary_path = output_dir / "phase2_pytest_run.txt"
summary_path.write_text(
    "Phase 2 pytest suite\n"
    f"Command: {sys.executable} -m pytest -v\n"
    f"Exit code: {result.returncode}\n\n"
    + result.stdout
    + ("\nSTDERR:\n" + result.stderr if result.stderr else ""),
    encoding="utf-8"
)
print("Pytest passed:", result.returncode == 0)
print("Execution record saved:", summary_path)


Pytest passed: True
Execution record saved: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\notebooks\outputs\phase2_pytest_run.txt
